# 12 — Cross-direction comparison: S/U vs Assistant Axis vs Refusal

With four directions in Llama 3.3 70B's residual space at layer 50, this notebook answers two questions:

1. **How aligned are they?** Pairwise cosine matrix tells you if they're orthogonal axes (independent phenomena), aligned (the same axis with different names), or partially overlapping.
2. **What features does each one decompose into?** Project each direction onto Goodfire's SAE → top-K features per direction → compute set overlap (Jaccard) and rank correlation between feature lists.

The four directions (set in config below):
- **S/U**: our exp10 fit on `llama33_70b` — instruction-source authority
- **Assistant Axis**: pre-computed, from `lu-christina/assistant-axis-vectors/llama-3.3-70b/assistant_axis.pt`
- **Default Vector**: also from the assistant-axis dataset — "baseline assistant persona" centroid
- **Refusal**: from `huihui-ai/Llama-3.3-70B-Instruct-abliterated/refusal_dir.pth`

All directions live in the same `d_model=8192` residual basis, so cosine and SAE-projection are directly comparable. **No model loading required**; only the SAE (4.3 GB) and the direction files (~1 MB each).

In [1]:
# Install (Colab / fresh box). Comment out if already in env.
# !pip install -q huggingface_hub torch numpy matplotlib seaborn

In [ ]:
# ---- Config ----
from pathlib import Path

# Layer at which the SAE-projection comparison runs (Goodfire's SAE is fixed at L50).
SAE_LAYER       = 50

# Goodfire SAE
SAE_REPO_ID     = 'Goodfire/Llama-3.3-70B-Instruct-SAE-l50'
SAE_FILENAME    = 'Llama-3.3-70B-Instruct-SAE-l50.pt'

# Local S/U NPZ — has 4 methods × 3 positions × 80 layers = 960 entries.
SUI_NPZ_PATH    = Path.cwd().parent / 'exp06_lamma' / 'directions.npz'
# (display_name, key format with {layer:03d}). Toggle with INCLUDE below.
SUI_VARIANTS = [
    ('sui_mm',         'mm_dir__response_last__layer_{layer:03d}'),
    ('sui_mm_raw',     'mm_raw__response_last__layer_{layer:03d}'),       # redundant w/ sui_mm after norm
    ('sui_pca_diff',   'pca_diff_dir__response_last__layer_{layer:03d}'),
    ('sui_pca_center', 'pca_center_dir__response_last__layer_{layer:03d}'),
]

# Apollo deception probes (Goldowsky-Dill et al. 2502.03407) — fit at LAYER 22, not 50.
# Files at third_party/apollo_deception_probes/<name>_detector.pt; saved with raw pickle, not torch.save.
APOLLO_PROBES   = ['instructed_pairs', 'roleplaying', 'descriptive', 'followup']
APOLLO_CACHE    = Path.cwd().parent / 'third_party' / 'apollo_deception_probes'
APOLLO_LAYER    = 22                                # the layer Apollo fit at

# Llama 3.3 70B has 80 transformer layers.
N_LAYERS        = 80
LAYERS_TO_SWEEP = list(range(N_LAYERS))

# Top-K features per direction (used by SAE-projection sections at SAE_LAYER).
TOP_K           = 25

INCLUDE = {
    # S/U variants — per-layer, all 4 methods at response_last
    'sui_mm':            True,
    'sui_mm_raw':        False,   # off — exactly equals sui_mm after unit-normalization
    'sui_pca_diff':      True,
    'sui_pca_center':    True,
    # external comparison directions (L50)
    'assistant_axis':    True,
    'default_vector':    True,
    'refusal':           True,
    # Apollo deception probes (L22). Off in main L50 matrix to avoid cross-layer noise;
    # the dedicated section below handles the L22-vs-L22 comparison instead.
    'apollo_instructed_pairs': False,
    'apollo_roleplaying':      False,
    'apollo_descriptive':      False,
    'apollo_followup':         False,
}

# Subset of directions used for the SAE-projection sections (cells 4 onward).
# Cosine matrix and per-layer sweep keep the full INCLUDE set; only SAE projection is filtered.
# The two S/U variants kept here (mm + pca_center) cover the contrast cleanly without redundancy:
#   sui_mm         — mean-difference (the canonical contrast direction)
#   sui_pca_center — leading PC of centered combined acts (sanity check; should give similar features)
# sui_pca_diff is dropped because it sits ~orthogonal to mm in cosine space, so its features
# would tell a separate story — keep it in the cosine matrix but out of SAE for now.
SAE_DIRECTION_NAMES = [
    'sui_mm',
    'sui_pca_center',
    'assistant_axis',
    'default_vector',
    'refusal',
]

print(f'main matrix: {sum(INCLUDE.values())} directions; layer sweep covers {len(LAYERS_TO_SWEEP)} layers')
print(f'SAE feature projection runs at L{SAE_LAYER} over {len(SAE_DIRECTION_NAMES)} directions: {SAE_DIRECTION_NAMES}')
print(f'Apollo probes (L{APOLLO_LAYER}) handled separately below')

### Method note — decoder cosine, not `encoder.forward(direction)`

We rank features per direction by `cosine(direction, W_dec[i])` rather than by passing the direction through the SAE encoder. Mayne et al. (NeurIPS 2024, *"Can SAEs be used to decompose and interpret steering vectors?"*) showed that direct encoding fails for steering vectors: their norms are out-of-distribution for the SAE (so the encoder bias dominates pre-activations), the SAE can only emit non-negative coefficients (so it can't represent negative projections), and TopK SAEs (like Goodfire's, k=121) silently mask all of this.

`W_dec[i]` is the residual-stream direction feature *i* writes when it fires, so cosine with `W_dec` cleanly answers "which features push the residual stream along this direction." We also compute encoder cosine as a side-by-side check; large disagreement between the two rankings is itself a signal worth flagging.

## 1 — Load all directions into a single (n_dirs, d_model) matrix

Normalizes each to unit norm. Skips any whose file is missing (with a warning) so the notebook still runs partially.

In [ ]:
import torch
import numpy as np
from huggingface_hub import hf_hub_download

DIRECTIONS = {}            # name -> (d_model,) unit tensor at SAE_LAYER  (used by SAE projection)
DIRECTIONS_PER_LAYER = {}  # name -> (n_layers, d_model) row-normalized   (used by per-layer cosine sweep)

def _normalize(t: torch.Tensor) -> torch.Tensor:
    return t / (t.norm() + 1e-10)

def _row_normalize(t: torch.Tensor) -> torch.Tensor:
    return t / (t.norm(dim=-1, keepdim=True) + 1e-10)

def _select_layer(t: torch.Tensor, layer: int, name: str) -> torch.Tensor:
    """Pick the row at `layer` from a 1D-or-2D tensor. 1D is returned as-is."""
    t = t.float().squeeze()
    if t.ndim == 2:
        print(f'  {name}: 2D tensor {tuple(t.shape)} — picking row {layer}')
        return t[layer]
    if t.ndim == 1:
        return t
    raise ValueError(f'{name}: unexpected tensor shape {tuple(t.shape)}')

def _maybe_per_layer(t: torch.Tensor, name: str) -> torch.Tensor | None:
    """Return row-normalized (n_layers, d_model) if t looks per-layer; else None."""
    t = t.float().squeeze()
    if t.ndim == 2 and t.shape[0] == N_LAYERS and t.shape[1] >= 1024:
        return _row_normalize(t)
    return None

# ---- our S/U directions (local NPZ — multiple variants, all per-layer) ----
arr = None
if SUI_NPZ_PATH.exists():
    arr = np.load(SUI_NPZ_PATH)
else:
    print(f'sui*           SKIP — no NPZ at {SUI_NPZ_PATH}')

if arr is not None:
    for sui_name, fmt in SUI_VARIANTS:
        if not INCLUDE.get(sui_name, False):
            continue
        per_layer, missing = [], []
        for L in range(N_LAYERS):
            k = fmt.format(layer=L)
            if k in arr.files:
                per_layer.append(torch.from_numpy(arr[k]).float())
            else:
                missing.append(k); per_layer.append(None)
        if missing:
            print(f'{sui_name:<14s} SKIP — missing keys (first: {missing[:2]})')
            continue
        stacked = torch.stack(per_layer)                          # (80, d_model)
        DIRECTIONS_PER_LAYER[sui_name] = _row_normalize(stacked)
        DIRECTIONS[sui_name]           = _normalize(stacked[SAE_LAYER])
        print(f'{sui_name:<14s} loaded per-layer ({tuple(stacked.shape)})')

# ---- assistant axis / default vector (HF) ----
for key, fname in [('assistant_axis', 'assistant_axis.pt'),
                   ('default_vector', 'default_vector.pt')]:
    if not INCLUDE.get(key, False): continue
    try:
        p = hf_hub_download(
            repo_id='lu-christina/assistant-axis-vectors',
            filename=f'llama-3.3-70b/{fname}',
            repo_type='dataset',
        )
        obj = torch.load(p, map_location='cpu', weights_only=False)
        DIRECTIONS[key] = _normalize(_select_layer(obj, SAE_LAYER, key))
        per_layer = _maybe_per_layer(obj, key)
        if per_layer is not None:
            DIRECTIONS_PER_LAYER[key] = per_layer
            print(f'{key:<14s} loaded per-layer ({tuple(per_layer.shape)})')
        else:
            shape = tuple(torch.as_tensor(obj).float().squeeze().shape)
            print(f'{key:<14s} loaded single ({shape}) — excluded from per-layer sweep')
    except Exception as e:
        print(f'{key:<14s} SKIP — {e}')

# ---- refusal direction (HF abliterated — typically a single vector) ----
if INCLUDE.get('refusal', False):
    try:
        p = hf_hub_download(
            repo_id='huihui-ai/Llama-3.3-70B-Instruct-abliterated',
            filename='refusal_dir.pth',
        )
        obj = torch.load(p, map_location='cpu', weights_only=False)
        DIRECTIONS['refusal'] = _normalize(_select_layer(obj, SAE_LAYER, 'refusal'))
        per_layer = _maybe_per_layer(obj, 'refusal')
        if per_layer is not None:
            DIRECTIONS_PER_LAYER['refusal'] = per_layer
            print(f'refusal        loaded per-layer ({tuple(per_layer.shape)})')
        else:
            shape = tuple(torch.as_tensor(obj).float().squeeze().shape)
            print(f'refusal        loaded single ({shape}) — excluded from per-layer sweep')
    except Exception as e:
        print(f'refusal        SKIP — {e}')

assert DIRECTIONS, 'no directions loaded — check INCLUDE flags and file paths.'

# Stack at SAE_LAYER for the cosine + SAE-projection sections below.
names = list(DIRECTIONS.keys())
M = torch.stack([DIRECTIONS[n] for n in names], dim=0)
print(f'\nat L{SAE_LAYER}: {len(names)} directions × d_model={M.shape[1]}')
print(f'names: {names}')
print(f'per-layer available for: {list(DIRECTIONS_PER_LAYER.keys())}')

assert all(d.shape == M[0].shape for d in DIRECTIONS.values())

## 2 — Pairwise cosine similarity

Quick read on whether the directions are orthogonal (cos ≈ 0), aligned (cos ≈ 1), or anti-aligned (cos ≈ -1). Diagonals are 1 by construction.

In [ ]:
cos_matrix = (M @ M.T).numpy()    # (n_dirs, n_dirs)

print('Pairwise cosine similarity:')
print(' ' * 18 + ' '.join(f'{n:>16s}' for n in names))
for i, n in enumerate(names):
    row = ' '.join(f'{cos_matrix[i, j]:+16.4f}' for j in range(len(names)))
    print(f'{n:>16s}  {row}')

# Plot heatmap
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(1.2 + 1.0 * len(names), 1.0 + 0.9 * len(names)))
    im = ax.imshow(cos_matrix, vmin=-1, vmax=1, cmap='RdBu_r')
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=30, ha='right')
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
    for i in range(len(names)):
        for j in range(len(names)):
            ax.text(j, i, f'{cos_matrix[i, j]:+.2f}', ha='center', va='center',
                    color='white' if abs(cos_matrix[i, j]) > 0.5 else 'black', fontsize=9)
    ax.set_title(f'Direction cosine (Llama 3.3 70B, L{SAE_LAYER})')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()
except ImportError:
    print('(matplotlib not available, skipping heatmap)')

## 2.5 — Per-layer pairwise cosine sweep

The single-layer cosine matrix above is at `SAE_LAYER` (50). But not every direction is meaningful at L50 — the S/U signal in particular varies sharply with depth, and persona/refusal axes can peak elsewhere. This cell computes pairwise cosine **at every layer** for the directions that have per-layer data.

**Eligibility (auto-detected at load):**
- `sui` — always per-layer (NPZ has 80 layers).
- `assistant_axis`, `default_vector` — per-layer iff the `.pt` is shape `(80, d_model)`. Single-vector files are excluded from the sweep but still appear in the L50 matrix above.
- `refusal` — typically a single vector picked at one "best" layer; usually excluded from the sweep.

The plot's red dashed line marks `SAE_LAYER` so you can read off where the per-layer values land relative to the L50 numbers shown above.

In [ ]:
# ---- Per-layer pairwise cosine ----
# Only directions whose source provides one vector per residual-stream layer.
sweep_names = list(DIRECTIONS_PER_LAYER.keys())
print(f'per-layer sweep over {len(sweep_names)} direction(s): {sweep_names}')

if len(sweep_names) >= 2:
    # (n_dirs, n_layers, d_model)
    sweep = torch.stack([DIRECTIONS_PER_LAYER[n] for n in sweep_names], dim=0)
    # Pairwise cosine at every layer: (n_dirs, n_dirs, n_layers)
    cos_per_layer = torch.einsum('inh,jnh->ijn', sweep, sweep).numpy()
    print(f'sweep shape: {tuple(sweep.shape)}  →  cos_per_layer: {cos_per_layer.shape}\n')

    print(f'pair summary  (mean / |max| @ best layer / value @ L{SAE_LAYER}):')
    for i in range(len(sweep_names)):
        for j in range(i + 1, len(sweep_names)):
            r = cos_per_layer[i, j]
            best = int(np.argmax(np.abs(r)))
            print(f'  {sweep_names[i]:>14s} × {sweep_names[j]:<14s}  '
                  f'mean={r.mean():+.3f}  |max|={np.abs(r).max():.3f} @ L{best:02d}  '
                  f'L{SAE_LAYER}={r[SAE_LAYER]:+.3f}')

    try:
        import matplotlib.pyplot as plt
        # Line plot — cosine vs layer per pair.
        fig, ax = plt.subplots(figsize=(11, 4.5))
        layers = np.array(LAYERS_TO_SWEEP)
        for i in range(len(sweep_names)):
            for j in range(i + 1, len(sweep_names)):
                ax.plot(layers, cos_per_layer[i, j, layers],
                        label=f'{sweep_names[i]} × {sweep_names[j]}', lw=1.5)
        ax.axhline(0, color='gray', lw=0.5)
        ax.axvline(SAE_LAYER, color='red', lw=0.5, ls='--', label=f'SAE L{SAE_LAYER}')
        ax.set_xlabel('residual-stream layer'); ax.set_ylabel('cosine similarity')
        ax.set_ylim(-1.05, 1.05)
        ax.set_title('Per-layer pairwise cosine — Llama 3.3 70B')
        ax.legend(loc='best', fontsize=8); ax.grid(alpha=0.3)
        plt.tight_layout(); plt.show()

        # Heatmap — pairs (rows) × layer (cols).
        if len(sweep_names) >= 2:
            pair_labels = [f'{sweep_names[i]} × {sweep_names[j]}'
                           for i in range(len(sweep_names))
                           for j in range(i + 1, len(sweep_names))]
            pair_data = np.stack([cos_per_layer[i, j, layers]
                                  for i in range(len(sweep_names))
                                  for j in range(i + 1, len(sweep_names))], axis=0)
            fig, ax = plt.subplots(figsize=(12, 0.5 + 0.5 * len(pair_labels)))
            im = ax.imshow(pair_data, vmin=-1, vmax=1, cmap='RdBu_r', aspect='auto',
                           extent=[layers[0] - 0.5, layers[-1] + 0.5, len(pair_labels) - 0.5, -0.5])
            ax.set_yticks(range(len(pair_labels))); ax.set_yticklabels(pair_labels)
            ax.set_xlabel('residual-stream layer')
            ax.axvline(SAE_LAYER, color='black', lw=0.7, ls='--')
            ax.set_title(f'Per-layer cosine — heatmap')
            plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
            plt.tight_layout(); plt.show()
    except ImportError:
        print('(matplotlib not available, skipping plots)')
else:
    print('need ≥2 per-layer directions to sweep; only assistant_axis/default_vector are likely candidates besides sui.')

## 2.6 — Apollo deception probes at L22 (apples-to-apples)

Apollo's LR probes were fit at **L22**, not L50. Putting them in the L50 cosine matrix would mix layers — the residual basis is shared, but the activation distributions aren't, so the cosines are hard to interpret.

This section does the comparison cleanly:
- Loads the four Apollo probes (`instructed_pairs`, `roleplaying`, `descriptive`, `followup`) via raw `pickle` (Apollo saves with `pickle.dump`, not `torch.save`).
- σ-rescales the LR weights from z-scored space back to the raw activation basis (`w_raw = w / scaler_scale`) so cosines are comparable to the mean-difference directions.
- Pulls each S/U variant **at L22** from `DIRECTIONS_PER_LAYER`.
- Builds the L22 cosine matrix.
- Adds a per-layer sweep `cos(sui_mm @ each layer × apollo @ L22)` so you can see whether S/U at *any* layer aligns with deception structure, not just at L22.

**Random-baseline expectation in `d=8192` is `|cos| ≈ 0.011`.** Anything below ~0.05 is noise; 0.05–0.15 is weak signal; above 0.2 is real overlap.

The four Apollo probes also serve as a useful sanity check on each other — `instructed_pairs` is the cleanest contrast and should be most distinct from the others if they're capturing different facets of deception, OR all four should cluster together if there's a single underlying deception axis.

In [ ]:
# ---- Apollo deception probes at L22 (apples-to-apples) ----
# Apollo probes are LR weights fit at residual-stream L22 in z-scored activation space.
# σ-rescale to put them in the raw basis, then compare to S/U variants at L22 (not L50).
import pickle, math

def _load_apollo(name: str):
    p = APOLLO_CACHE / f'{name}_detector.pt'
    if not p.exists():
        print(f'  apollo_{name:<18s} not found at {p}')
        return None
    with open(p, 'rb') as f:
        obj = pickle.load(f)
    w = obj['directions'].squeeze(0).float()                     # (d_model,)
    if obj.get('normalize', False) and 'scaler_scale' in obj:
        w = w / obj['scaler_scale'].squeeze(0).float()           # back to raw-act basis
    return w

apollo_dirs = {}
for name in APOLLO_PROBES:
    w = _load_apollo(name)
    if w is None: continue
    apollo_dirs[f'apollo_{name}'] = _normalize(w)
    print(f'  apollo_{name:<18s} loaded (fit @ L{APOLLO_LAYER})')

if apollo_dirs:
    # Slice every per-layer direction at APOLLO_LAYER for apples-to-apples comparison.
    l22_mat = {n: DIRECTIONS_PER_LAYER[n][APOLLO_LAYER] for n in DIRECTIONS_PER_LAYER}
    l22_mat.update(apollo_dirs)
    l22_names = list(l22_mat.keys())
    L22 = torch.stack([l22_mat[n] for n in l22_names], dim=0)
    cos_l22 = (L22 @ L22.T).numpy()

    print(f'\nL{APOLLO_LAYER} cosine matrix ({len(l22_names)} directions):')
    print(' ' * 22 + ' '.join(f'{n:>22s}' for n in l22_names))
    for i, n in enumerate(l22_names):
        row = ' '.join(f'{cos_l22[i, j]:+22.4f}' for j in range(len(l22_names)))
        print(f'{n:>22s}  {row}')

    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(1.2 + 1.0 * len(l22_names), 1.0 + 0.9 * len(l22_names)))
        im = ax.imshow(cos_l22, vmin=-1, vmax=1, cmap='RdBu_r')
        ax.set_xticks(range(len(l22_names))); ax.set_xticklabels(l22_names, rotation=30, ha='right')
        ax.set_yticks(range(len(l22_names))); ax.set_yticklabels(l22_names)
        for i in range(len(l22_names)):
            for j in range(len(l22_names)):
                ax.text(j, i, f'{cos_l22[i, j]:+.2f}', ha='center', va='center',
                        color='white' if abs(cos_l22[i, j]) > 0.5 else 'black', fontsize=9)
        ax.set_title(f'Direction cosine @ L{APOLLO_LAYER} (Apollo fit layer)')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        plt.tight_layout(); plt.show()
    except ImportError:
        pass

    print(f'\nrandom-baseline |cos| in d=8192 ≈ {1/math.sqrt(8192):.4f}')
    print('< ~0.05 noise   |   0.05–0.15 weak   |   > 0.2 real overlap')

    # Per-layer cos(sui_mm × apollo_*) — does S/U at any other layer pick up deception structure?
    if 'sui_mm' in DIRECTIONS_PER_LAYER:
        sui_layers = DIRECTIONS_PER_LAYER['sui_mm']               # (80, d_model)
        apollo_stack = torch.stack(list(apollo_dirs.values()), dim=0)  # (n_apollo, d_model)
        cos_sweep = (sui_layers @ apollo_stack.T).numpy()         # (80, n_apollo)

        try:
            fig, ax = plt.subplots(figsize=(11, 4.5))
            for k, n in enumerate(apollo_dirs.keys()):
                ax.plot(LAYERS_TO_SWEEP, cos_sweep[LAYERS_TO_SWEEP, k], label=n, lw=1.5)
            ax.axhline(0, color='gray', lw=0.5)
            ax.axvline(APOLLO_LAYER, color='red', lw=0.7, ls='--', label=f'Apollo fit L{APOLLO_LAYER}')
            ax.axvline(SAE_LAYER,    color='blue', lw=0.5, ls=':',  label=f'SAE L{SAE_LAYER}')
            ax.set_xlabel('S/U residual-stream layer (sui_mm)'); ax.set_ylabel('cosine')
            ax.set_ylim(-0.5, 0.5)
            ax.set_title('cos(sui_mm @ each layer × apollo @ L22)')
            ax.legend(loc='best', fontsize=8); ax.grid(alpha=0.3)
            plt.tight_layout(); plt.show()
        except NameError:
            pass

        for k, n in enumerate(apollo_dirs.keys()):
            best = int(np.argmax(np.abs(cos_sweep[:, k])))
            print(f'  sui_mm × {n:<28s}  |max|={np.abs(cos_sweep[:, k]).max():.3f} @ L{best:02d}  '
                  f'L{APOLLO_LAYER}={cos_sweep[APOLLO_LAYER, k]:+.3f}  L{SAE_LAYER}={cos_sweep[SAE_LAYER, k]:+.3f}')
else:
    print('no Apollo probes loaded — check APOLLO_CACHE path.')

## 3 — Load Goodfire SAE (4.3 GB)

Same loader as notebook 11. Extracts both `W_enc` and `W_dec` reoriented to `(d_sae, d_model)` so each row is one feature.

In [ ]:
sae_path = hf_hub_download(repo_id=SAE_REPO_ID, filename=SAE_FILENAME)
sae_obj = torch.load(sae_path, map_location='cpu', weights_only=False)
sd = sae_obj.state_dict() if hasattr(sae_obj, 'state_dict') else sae_obj

def _find(state_dict, candidates):
    for c in candidates:
        if c in state_dict:
            return state_dict[c]
    raise KeyError(f'none of {candidates} found; first keys: {list(state_dict.keys())[:8]}')

W_enc = _find(sd, ['encoder.weight', 'W_enc', 'enc.weight', 'W_in']).float()
W_dec = _find(sd, ['decoder.weight', 'W_dec', 'dec.weight', 'W_out']).float()

# Reorient both to (d_sae, d_model). nn.Linear stores (out, in); decoder is
# Linear(d_sae -> d_model) so its weight is (d_model, d_sae) and needs a transpose.
if W_dec.shape[0] != W_enc.shape[0]:
    if W_dec.shape[1] == W_enc.shape[0]:
        W_dec = W_dec.T
    else:
        raise RuntimeError(
            f'W_dec shape {tuple(W_dec.shape)} not compatible with W_enc {tuple(W_enc.shape)}'
        )

d_sae, d_model = W_enc.shape
assert W_dec.shape == (d_sae, d_model)
print(f'W_enc: ({d_sae}, {d_model})  W_dec: {tuple(W_dec.shape)}')
assert d_model == M.shape[1], f'SAE d_model={d_model} != direction d_model={M.shape[1]}'

## 4 — Project every direction onto SAE features (decoder cosine)

Produces `(n_dirs, d_sae)` matrices for both decoder and encoder cosine. The decoder matrix is the primary one used downstream; the encoder matrix is reported alongside as a consistency check.

In [ ]:
# ---- Filter to the SAE-projection subset ----
# Re-binds `names`, `M`, `cos_matrix` to the directions you actually want to project through
# the SAE. The L50 cosine matrix and per-layer sweep above already ran on the full INCLUDE set,
# so this rebinding only affects sections 4–8 below.
sae_keep = [n for n in SAE_DIRECTION_NAMES if n in DIRECTIONS]
sae_missing = [n for n in SAE_DIRECTION_NAMES if n not in DIRECTIONS]
if sae_missing:
    print(f'NOTE: SAE_DIRECTION_NAMES requested but not loaded: {sae_missing}')

names = sae_keep
M = torch.stack([DIRECTIONS[n] for n in names], dim=0)
cos_matrix = (M @ M.T).numpy()
print(f'SAE projection over {len(names)} direction(s): {names}\n')

with torch.no_grad():
    dec_norms = W_dec.norm(dim=1) + 1e-10            # (d_sae,)
    enc_norms = W_enc.norm(dim=1) + 1e-10            # (d_sae,)
    feat_sims_dec = (M @ W_dec.T) / dec_norms        # (n_dirs, d_sae)  PRIMARY
    feat_sims_enc = (M @ W_enc.T) / enc_norms        # (n_dirs, d_sae)  secondary

# Backward-compat alias for downstream cells / saved artifacts.
feat_sims = feat_sims_dec

print(f'feat_sims_dec shape: {tuple(feat_sims_dec.shape)}')
print(f'\nper-direction stats (DECODER cosine):')
for i, n in enumerate(names):
    s = feat_sims_dec[i]
    print(f'  {n:>16s}   max={s.max():+.4f}  min={s.min():+.4f}  mean={s.mean():+.4f}  std={s.std():.4f}')
print(f'\nper-direction stats (ENCODER cosine):')
for i, n in enumerate(names):
    s = feat_sims_enc[i]
    print(f'  {n:>16s}   max={s.max():+.4f}  min={s.min():+.4f}  mean={s.mean():+.4f}  std={s.std():.4f}')

# Per-direction Pearson correlation between decoder and encoder rankings.
print(f'\nper-direction corr(decoder cos, encoder cos):')
for i, n in enumerate(names):
    c = torch.corrcoef(torch.stack([feat_sims_dec[i], feat_sims_enc[i]]))[0, 1].item()
    print(f'  {n:>16s}   corr = {c:+.4f}')

In [ ]:
# Top-K features per direction by DECODER cosine (positive side).
top_k_pos = {}   # name -> {'idx': [...], 'sim_dec': [...], 'sim_enc': [...]}
for i, n in enumerate(names):
    tk = torch.topk(feat_sims_dec[i], TOP_K, largest=True)
    idx_list = tk.indices.tolist()
    top_k_pos[n] = {
        'idx': idx_list,
        'sim_dec': tk.values.tolist(),
        'sim_enc': [feat_sims_enc[i, j].item() for j in idx_list],
    }

print(f'Top-{TOP_K} feature IDs per direction (by decoder cosine):')
for n in names:
    ids = ' '.join(f'{i:6d}' for i in top_k_pos[n]['idx'][:10])
    print(f'  {n:>16s}   {ids} ...')

## 5 — Overlap between top-K feature sets (Jaccard)

For each pair of directions, fraction of top-K features they share. High Jaccard = directions decompose into overlapping features (likely related). Low Jaccard = orthogonal feature stories.

In [ ]:
n = len(names)
jaccard = np.zeros((n, n))
for i in range(n):
    a = set(top_k_pos[names[i]]['idx'])
    for j in range(n):
        b = set(top_k_pos[names[j]]['idx'])
        union = len(a | b)
        jaccard[i, j] = len(a & b) / union if union else 0.0

print(f'Jaccard overlap of top-{TOP_K} feature sets:')
print(' ' * 18 + ' '.join(f'{n:>16s}' for n in names))
for i in range(len(names)):
    row = ' '.join(f'{jaccard[i, j]:16.3f}' for j in range(len(names)))
    print(f'{names[i]:>16s}  {row}')

try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(2 + 1.6 * len(names), 1 + 0.9 * len(names)))
    for ax, mat, title, vmin, vmax, cmap in [
        (axes[0], cos_matrix, 'cosine (direction-space)', -1, 1, 'RdBu_r'),
        (axes[1], jaccard,    f'Jaccard top-{TOP_K} features', 0, 1, 'YlGn'),
    ]:
        im = ax.imshow(mat, vmin=vmin, vmax=vmax, cmap=cmap)
        ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=30, ha='right')
        ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
        for i in range(len(names)):
            for j in range(len(names)):
                v = mat[i, j]
                ax.text(j, i, f'{v:+.2f}' if vmin < 0 else f'{v:.2f}', ha='center', va='center',
                        color='white' if (abs(v) > 0.5 if vmin < 0 else v > 0.5) else 'black',
                        fontsize=9)
        ax.set_title(title)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()
except ImportError:
    print('(matplotlib not available)')

## 6 — Per-direction top-features grid (with Neuronpedia URLs)

For each direction, the top-K feature IDs and their cosine to that direction. Click through Neuronpedia URLs (or run notebook 11's auto-interp cell on each direction individually) to read descriptions.

In [ ]:
NEURONPEDIA_MODEL_SLUG = 'llama3.3-70b'   # confirm on neuronpedia.org
NEURONPEDIA_SAE_SLUG   = '50-goodfire-sae-l50'  # placeholder

for n in names:
    print(f'\n=== {n} ===')
    rows = zip(top_k_pos[n]['idx'], top_k_pos[n]['sim_dec'], top_k_pos[n]['sim_enc'])
    for rank, (idx, sd_, se_) in enumerate(rows):
        url = f'https://www.neuronpedia.org/{NEURONPEDIA_MODEL_SLUG}/{NEURONPEDIA_SAE_SLUG}/{idx}'
        print(f'  #{rank+1:2d}  feature_{idx:6d}   cos_dec={sd_:+.4f}  cos_enc={se_:+.4f}   {url}')

## 7 — Shared vs unique features

For each pair of directions, list the features they have in common in the top-K — and the features unique to each. This is the most readable summary: which features tell the *shared* story between (e.g.) S/U and Refusal, vs which features distinguish them.

In [ ]:
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a_name, b_name = names[i], names[j]
        a = set(top_k_pos[a_name]['idx'])
        b = set(top_k_pos[b_name]['idx'])
        shared = sorted(a & b)
        only_a = sorted(a - b)
        only_b = sorted(b - a)
        print(f'\n--- {a_name}  vs  {b_name} ---')
        print(f'  shared ({len(shared):3d}): {shared[:15]}')
        print(f'  only {a_name:>14s} ({len(only_a):3d}): {only_a[:8]}')
        print(f'  only {b_name:>14s} ({len(only_b):3d}): {only_b[:8]}')

## 8 — Save outputs

Writes the cosine matrix, Jaccard matrix, per-direction top-K features, and metadata to disk so downstream analysis / plots don't re-run the SAE load.

In [ ]:
import json

OUT_DIR = Path.cwd().parent / 'exp_direction_comparison' / 'llama33_70b_l50'
OUT_DIR.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    OUT_DIR / 'comparison.npz',
    names=np.array(names),
    cos_matrix=cos_matrix,
    jaccard=jaccard,
    feat_sims_dec=feat_sims_dec.numpy(),
    feat_sims_enc=feat_sims_enc.numpy(),
)

report = {
    'sae_repo': SAE_REPO_ID,
    'sae_layer': SAE_LAYER,
    'd_model': int(d_model),
    'd_sae': int(d_sae),
    'top_k': int(TOP_K),
    'directions': names,
    'projection_method': 'cosine(direction, W_dec[i]) — primary; W_enc cosine reported alongside',
    'pairwise_cosine': {
        f'{names[i]}__vs__{names[j]}': float(cos_matrix[i, j])
        for i in range(len(names)) for j in range(i + 1, len(names))
    },
    'pairwise_jaccard_topk_decoder': {
        f'{names[i]}__vs__{names[j]}': float(jaccard[i, j])
        for i in range(len(names)) for j in range(i + 1, len(names))
    },
    'top_k_features': {n: top_k_pos[n] for n in names},
}
with open(OUT_DIR / 'report.json', 'w') as f:
    json.dump(report, f, indent=2)

print(f'wrote {OUT_DIR / "comparison.npz"}')
print(f'wrote {OUT_DIR / "report.json"}')